# OpenBind blind challenge

Everything in one place. Sections A and B are setup and run every session.
After that, run only the section you need.

| | |
| --- | --- |
| **A** | Setup, every session |
| **B** | Status, how many of 400 are done |
| **C** | The main run, 400 compounds, one pose each — DONE 19 Nov |
| **D** | Build the register of confidence scores — DONE 19 Nov |
| **E** | Package the submission zip |
| **F** | Replicates: timing test, then the full replicate run |

Runtime > Change runtime type > **L4**, tick **High-RAM**.

## A. Setup — run these four every session

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil; print(f'{psutil.virtual_memory().total/1e9:.1f} GB RAM')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/openbind-blind-challenge')
DRIVE_PRED = ROOT / 'predictions'; DRIVE_PRED.mkdir(parents=True, exist_ok=True)
LOCAL = pathlib.Path('/content/preds'); LOCAL.mkdir(exist_ok=True)
print(len(list((ROOT/'data/queries').glob('batch_*.json'))), 'batch files')

In [ ]:
!pip -q install openfold3 awscli

In [ ]:
CKPT_DRIVE = ROOT / 'weights' / 'of3-ob-2025-06-30-174k.pt'
CKPT_DRIVE.parent.mkdir(exist_ok=True)
if not CKPT_DRIVE.exists():
    !aws s3 cp s3://openfold3-data/openfold3-parameters/of3-ob-2025-06-30-174k.pt "{CKPT_DRIVE}" --no-sign-request
CKPT = '/content/ckpt.pt'
!cp "{CKPT_DRIVE}" {CKPT}

(ROOT / 'runner.yml').write_text('''output_writer_settings:
  structure_format: pdb
model_update:
  presets:
    - predict
    - low_mem
''')
print('ready')

## B. Status

In [ ]:
have = sorted(p.name for p in DRIVE_PRED.glob('blind_challenge_*') if any(p.rglob('*_model.pdb')))
print(f'{len(have)}/400 complete')
missing = [f'blind_challenge_{i:03d}' for i in range(1, 401) if f'blind_challenge_{i:03d}' not in have]
if missing: print('missing:', missing[:20])

## C. The main run — already done

Only re-run this if section B shows compounds missing. It skips whatever is
already saved on Drive.

In [ ]:
import json, time

def saved():
    return {p.name for p in DRIVE_PRED.glob('blind_challenge_*') if any(p.rglob('*_model.pdb'))}

for b in sorted((ROOT / 'data/queries').glob('batch_*.json')):
    ids = list(json.load(open(b))['queries'])
    if all(i in saved() for i in ids):
        print(f'{b.stem}: done'); continue
    print(f'\n===== {b.stem} =====', flush=True)
    t0 = time.time()
    !run_openfold predict --query-json={b} --output-dir={LOCAL} --inference-ckpt-path={CKPT} --runner-yaml={ROOT}/runner.yml --num-diffusion-samples=1 --use-msa-server=True
    !cp -r {LOCAL}/blind_challenge_* "{DRIVE_PRED}/"
    print(f'{b.stem}: {(time.time()-t0)/60:.1f} min, {len(saved())}/400 saved', flush=True)

## D. The register — already done

Writes `analysis/register.csv`: one row per compound, every confidence field.

In [ ]:
import json, csv

def flatten(d):
    flat = {}
    for k, v in d.items():
        if isinstance(v, dict):
            for k2, v2 in v.items():
                clean = k2.replace('(','').replace(')','').replace(', ','_').replace(' ','')
                flat[f'{k}_{clean}'] = v2
        else:
            flat[k] = v
    return flat

recs = []
for d in sorted(DRIVE_PRED.glob('blind_challenge_*')):
    conf = sorted(d.rglob('*_confidences_aggregated.json'))
    pdbs = sorted(d.rglob('*_model.pdb'))
    if not conf or not pdbs:
        print('incomplete:', d.name); continue
    rec = {'id': d.name}
    rec.update(flatten(json.load(open(conf[0]))))
    rec['ligand_atoms'] = sum(1 for l in pdbs[0].read_text().splitlines() if l.startswith('HETATM'))
    recs.append(rec)

out = ROOT / 'analysis'; out.mkdir(exist_ok=True)
with (out/'register.csv').open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(recs[0])); w.writeheader(); w.writerows(recs)
print(f'{len(recs)} compounds -> analysis/register.csv')

## E. Package the submission

Renames each prediction to `blind_challenge_XXX.pdb` at the zip root, and refuses
to build the archive unless all 400 are present. Download `submission.zip` from
Drive and upload it through the HuggingFace Space.

In [ ]:
import zipfile, shutil, pathlib

stage = pathlib.Path('/content/submission'); shutil.rmtree(stage, ignore_errors=True); stage.mkdir()
missing = []
for i in range(1, 401):
    cid = f'blind_challenge_{i:03d}'
    hits = sorted((DRIVE_PRED / cid).rglob('*_model.pdb'))
    if not hits:
        missing.append(cid); continue
    shutil.copy(hits[0], stage / f'{cid}.pdb')

print(f'{len(list(stage.glob("*.pdb")))} staged, {len(missing)} missing')
if missing:
    print('MISSING:', missing)
else:
    z = ROOT / 'submission.zip'
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(stage.glob('*.pdb')):
            zf.write(p, arcname=p.name)
    print(f'wrote {z}, {z.stat().st_size/1e6:.1f} MB')

## F. Replicates

Run the same compounds several times from different seeds, to see how far apart
the answers land. Section F1 is a timing test on three compounds. Do not run F3
until F1 tells you what it costs.

### F1. Timing test

In [ ]:
import json, pathlib, time

q = json.load(open(ROOT / 'data/queries/batch_000.json'))['queries']
three = {k: q[k] for k in list(q)[:3]}
pathlib.Path('/content/three.json').write_text(json.dumps({'queries': three}, indent=1))

t0 = time.time()
!run_openfold predict --query-json=/content/three.json --output-dir=/content/seedtest --inference-ckpt-path={CKPT} --runner-yaml={ROOT}/runner.yml --num-model-seeds=5 --use-msa-server=True
print(f'\n{(time.time()-t0)/60:.1f} min for 3 compounds at 5 seeds')
print('   baseline was 2.8 min for 3 at 1 seed')

### F2. What the output looks like

In [ ]:
import pathlib
for p in sorted(pathlib.Path('/content/seedtest').rglob('*_model.pdb')):
    print(p.relative_to('/content/seedtest'))

### F3. The replicate run

Do not run this yet. The number of compounds and seeds depends on what F1 costs,
and the choice is recorded in `notes/PREREGISTRATION.md` before it runs.

In [ ]:
# filled in once F1 has answered the cost question